# This is a Jupyter Notebook

In [1]:
%%time

import scikitplot as sp
sp.__version__

CPU times: user 590 ms, sys: 310 ms, total: 901 ms
Wall time: 7.47 s


'0.5.dev0+git.20260311.78a6c29'

In [ ]:
# First we download the media preproccess libraries (text, image, audio or video).
# pip install nltk gensim langdetect faster-whisper openai-whisper pytesseract youtube-transcript-api
# sudo apt-get install tesseract-ocr
from scikitplot import corpus

In [3]:
import os
import json
import sys
import textwrap
from pathlib import Path

In [4]:

from scikitplot.corpus._schema import (
    CorpusDocument,
    SourceType,
    MatchMode,
    SectionType,
    ChunkingStrategy,
    _PROMOTED_RAW_KEYS,
)
from scikitplot.corpus._base import DocumentReader, DefaultFilter
from scikitplot.corpus._normalizers import (
    NormalizationPipeline,
    UnicodeNormalizer,
    WhitespaceNormalizer,
)
from scikitplot.corpus._enrichers._nlp_enricher import NLPEnricher, EnricherConfig
from scikitplot.corpus._similarity._similarity import (
    SimilarityIndex,
    SearchConfig,
    SearchResult,
)
from scikitplot.corpus._adapters import (
    to_langchain_documents,
    to_langgraph_state,
    to_mcp_resources,
    to_mcp_tool_result,
    to_huggingface_dataset,
    to_rag_tuples,
    to_jsonl,
    MCPCorpusServer,
)

In [5]:

# ===========================================================================
# HELPER: print section banners
# ===========================================================================

def banner(title: str, char: str = "=") -> None:
    line = char * 72
    print(f"\n{line}\n  {title}\n{line}\n")

def mini_banner(title: str) -> None:
    print(f"\n  --- {title} ---\n")

def show_doc(doc: CorpusDocument, index: int = 0) -> None:
    """Pretty-print a single CorpusDocument."""
    text_preview = doc.text[:100].replace("\n", " ")
    norm_preview = (doc.normalized_text or "")[:80].replace("\n", " ")
    print(f"  [{index:3d}] doc_id={doc.doc_id[:12]}…  source_type={doc.source_type}")
    print(f"        text: {text_preview!r}…")
    if norm_preview:
        print(f"        norm: {norm_preview!r}…")
    if doc.tokens:
        print(f"        tokens({len(doc.tokens)}): {doc.tokens[:8]}…")
    if doc.keywords:
        print(f"        keywords: {doc.keywords[:6]}")
    if doc.timecode_start is not None:
        print(f"        timecode: {doc.timecode_start:.1f}s – {doc.timecode_end:.1f}s")
    if doc.confidence is not None:
        print(f"        confidence: {doc.confidence:.3f}")
    if doc.page_number is not None:
        print(f"        page: {doc.page_number}")



In [6]:

# ===========================================================================
# PHASE 1: INGEST — Process all 5 source types via DocumentReader
# ===========================================================================

banner("PHASE 1: INGEST — 5 Source Types via DocumentReader")

all_documents: list[CorpusDocument] = []
source_log: list[dict] = []

# --- Source ①: Web Article (HTML) ---
# In production: DocumentReader.from_url("https://www.who.int/europe/news/item/...")
# Here we use the local text proxy (same content)
mini_banner("Source ①: Web Article (text proxy for HTML URL)")
try:
    if Path("who_health_care_article.txt").exists():
        # from local raw html page
        reader = DocumentReader.create(
            Path("who_health_care_article.txt"),
            source_type=SourceType.WEB,
            source_title="Out-of-pocket payments for health care unaffordable for millions in Europe",
            source_author="WHO Regional Office for Europe",
            source_date="2023-12-12",
            collection_id="who-greece-financial-protection",
        )
    else:
        # from url raw html page DocumentReader.from_url for raw html
        reader = DocumentReader.from_url(
            "https://www.who.int/europe/news/item/12-12-2023-out-of-pocket-payments-for-primary-health-care-unaffordable-for-millions-in-europe-new-who-report-shows",
        )
    docs = list(reader.get_documents())
    all_documents.extend(docs)
    source_log.append({"type": "web_article", "n_docs": len(docs), "status": "OK"})
    print(f"  ✓ Web article: {len(docs)} chunks ingested")
    if docs:
        show_doc(docs[0], 0)
except Exception as e:
    source_log.append({"type": "web_article", "n_docs": 0, "status": f"ERROR: {e}"})
    print(f"  ✗ Web article: {e}")


  PHASE 1: INGEST — 5 Source Types via DocumentReader


  --- Source ①: Web Article (text proxy for HTML URL) ---

2026-03-17 19:24:13.833733: I scikitplot.corpus._readers._web 133692742501952 _web.py:448:get_raw_chunks] WebReader: fetching https://www.who.int/europe/news/item/12-12-2023-out-of-pocket-payments-for-primary-health-care-unaffordable-for-millions-in-europe-new-who-report-shows.
2026-03-17 19:24:13.965623: I scikitplot.corpus._readers._web 133692742501952 _web.py:490:get_raw_chunks] WebReader: fetched https://www.who.int/europe/news/item/12-12-2023-out-of-pocket-payments-for-primary-health-care-unaffordable-for-millions-in-europe-new-who-report-shows (139485 bytes).
2026-03-17 19:24:14.089766: I scikitplot.corpus._readers._web 133692742501952 _web.py:518:get_raw_chunks] WebReader: extracted 241 elements from https://www.who.int/europe/news/item/12-12-2023-out-of-pocket-payments-for-primary-health-care-unaffordable-for-millions-in-europe-new-who-report-shows.
2026-03-17 19:

In [7]:
docs[60].text

'Financial hardship caused by out-of-pocket payments for medicines, medical products such as hearing aids, and dental care affects millions of people, even in Europe’s richest countries, reveals a new WHO report. On International Universal Health Coverage (UHC) Day\xa02023, the report “Can people afford to pay for health care? Evidence on financial protection in 40 countries in Europe” highlights that out-of-pocket payments push between 1% and 12% of households into poverty or make them poorer.'

In [8]:
all_documents[60].text

'Financial hardship caused by out-of-pocket payments for medicines, medical products such as hearing aids, and dental care affects millions of people, even in Europe’s richest countries, reveals a new WHO report. On International Universal Health Coverage (UHC) Day\xa02023, the report “Can people afford to pay for health care? Evidence on financial protection in 40 countries in Europe” highlights that out-of-pocket payments push between 1% and 12% of households into poverty or make them poorer.'

In [9]:

# --- Source ②: YouTube Transcript ---
# In production: DocumentReader.from_url("https://youtu.be/rwPISgZcYIk")
mini_banner("Source ②: YouTube Transcript (text proxy)")
try:
    if Path("who_video_transcript.txt").exists():
        # TODO: Not implemented from local for raw video embedding support
        # from local video Transcript
        reader = DocumentReader.create(
            Path("who_video_transcript.txt"),
            source_type=SourceType.VIDEO,
            source_title="Can people afford to pay for health care? WHO Europe",
            collection_id="who-greece-financial-protection",
        )
    else:
        # TODO: Not implemented pdf from DocumentReader.from_url for raw video embedding support
        # from url video Transcript
        reader = DocumentReader.from_url("https://youtu.be/rwPISgZcYIk")
    docs = list(reader.get_documents())
    all_documents.extend(docs)
    source_log.append({"type": "youtube_transcript", "n_docs": len(docs), "status": "OK"})
    print(f"  ✓ YouTube transcript: {len(docs)} chunks ingested")
    if docs:
        show_doc(docs[0], 0)
except Exception as e:
    source_log.append({"type": "youtube_transcript", "n_docs": 0, "status": f"ERROR: {e}"})
    print(f"  ✗ YouTube transcript: {e}")



  --- Source ②: YouTube Transcript (text proxy) ---

2026-03-17 19:24:16.270640: I scikitplot.corpus._readers._web 133692742501952 _web.py:727:get_raw_chunks] YouTubeReader: fetching transcript for video rwPISgZcYIk.
2026-03-17 19:24:17.287414: I scikitplot.corpus._readers._web 133692742501952 _web.py:783:get_raw_chunks] YouTubeReader: using auto_generated en transcript for video rwPISgZcYIk.
2026-03-17 19:24:17.388982: I scikitplot.corpus._readers._web 133692742501952 _web.py:799:get_raw_chunks] YouTubeReader: retrieved 109 cues for video rwPISgZcYIk.
2026-03-17 19:24:17.395706: I scikitplot.corpus._base 133692742501952 _base.py:836:get_documents] https://youtu.be/rwPISgZcYIk: yielded 101 documents, omitted 2 (filter/empty).
  ✓ YouTube transcript: 101 chunks ingested
  [  0] doc_id=c9c3579363e5…  source_type=video
        text: '[Music] can people afford to pay for healthcare'…
        timecode: 1.0s – 11.2s


In [10]:
docs[0].text

'[Music] can people afford to pay for healthcare'

In [11]:

# --- Source ③: PDF Report ---
# In production: fetch PDF from URL then DocumentReader.create("report.pdf")
mini_banner("Source ③: PDF Report (text proxy)")
try:
    if Path("WHO-EURO-2025-12555-52329-80560-eng.pdf").exists():
        # from local pdf
        reader = DocumentReader.create(
            Path("WHO-EURO-2025-12555-52329-80560-eng.pdf"),
            source_type=SourceType.RESEARCH,
            source_title="Financial Protection Review: Greece Summary",
            source_author="WHO Barcelona Office",
            source_date="2023-01-01",
            doi="10.2307/who-greece-fp-2023",
            collection_id="who-greece-financial-protection",
        )
    else:
        # TODO: Not implemented pdf from DocumentReader.from_url pdf url link
        reader = DocumentReader.from_url(
            "https://iris.who.int/server/api/core/bitstreams/7ad66865-7f23-4485-8cf5-7b3d78bdf4f9/content",
        )
        pass
    docs = list(reader.get_documents())
    all_documents.extend(docs)
    source_log.append({"type": "pdf_report", "n_docs": len(docs), "status": "OK"})
    print(f"  ✓ PDF report: {len(docs)} chunks ingested")
    if docs:
        show_doc(docs[0], 0)
except Exception as e:
    source_log.append({"type": "pdf_report", "n_docs": 0, "status": f"ERROR: {e}"})
    print(f"  ✗ PDF report: {e}")



  --- Source ③: PDF Report (text proxy) ---

2026-03-17 19:24:18.485113: I scikitplot.corpus._readers._pdf 133692742501952 _pdf.py:402:get_raw_chunks] PDFReader: opening WHO-EURO-2025-12555-52329-80560-eng.pdf (6 page(s), backend=auto).
2026-03-17 19:24:18.916977: I scikitplot.corpus._readers._pdf 133692742501952 _pdf.py:430:get_raw_chunks] PDFReader: finished WHO-EURO-2025-12555-52329-80560-eng.pdf — 6 page(s) yielded text.
2026-03-17 19:24:18.918779: I scikitplot.corpus._base 133692742501952 _base.py:836:get_documents] WHO-EURO-2025-12555-52329-80560-eng.pdf: yielded 6 documents, omitted 0 (filter/empty).
  ✓ PDF report: 6 chunks ingested
  [  0] doc_id=9a1daab39190…  source_type=research
        text: 'Can people afford  to pay for health care?  New evidence on  financial protection  in Greece: summar'…
        page: 0


In [12]:
docs[0].text

'Can people afford \nto pay for health care? \nNew evidence on \nfinancial protection \nin Greece: summary\nThis review assesses the extent to which people in \nGreece face financial barriers to access or experience \nfinancial hardship (impoverishing or catastrophic health \nspending) when they use health care. It covers the period \nbetween 2008 and 2025, using data from household \nbudget surveys carried out from 2008 to 2023 (the latest \navailable year), data on unmet need for health care up \nto 2024 (the latest available year) and information on \ncoverage policy (population coverage, service coverage \nand user charges) up to May 2025 (UHC watch, 2025).\nIn 2023 3% of households were impoverished or further \nimpoverished after out-of-pocket payments (data not \nshown) and almost 10% of households experienced \ncatastrophic health spending, up from around 7%  \nin 2008 (Fig. 1).\nCatastrophic health spending is consistently heavily \nconcentrated in the poorest consumption quin

In [13]:
# pip install pytesseract
# sudo apt-get install tesseract-ocr

In [14]:

# --- Source ④: Document Scan (Image OCR) ---
mini_banner("Source ④: Document Scan (JPG → OCR)")
image_path = Path("WHO-EURO-2025-12555-52329-80560-eng.pdf.jpg")
try:
    if image_path.exists():
        # TODO: Not implemented image from local
        reader = DocumentReader.create(
            image_path,
            source_type=SourceType.IMAGE,
            source_title="WHO Greece Report — Page 1 Scan",
            collection_id="who-greece-financial-protection",
        )
    else:
        # TODO: Not implemented for image DocumentReader.from_url image url link
        reader = DocumentReader.from_url(
            "https://iris.who.int/server/api/core/bitstreams/d57241c0-512d-4cfc-9ead-91a83eea83f0/content",
        )
        pass
    docs = list(reader.get_documents())
    all_documents.extend(docs)
    source_log.append({"type": "image_ocr", "n_docs": len(docs), "status": "OK"})
    print(f"  ✓ Image OCR: {len(docs)} chunks ingested")
    if docs:
        show_doc(docs[0], 0)
except ImportError as e:
    source_log.append({"type": "image_ocr", "n_docs": 0, "status": f"SKIP (dep missing): {e}"})
    print(f"  ⚠ Image OCR skipped (dependency not installed): {type(e).__name__}")
    print(f"    In production: pip install pytesseract Pillow")
except Exception as e:
    source_log.append({"type": "image_ocr", "n_docs": 0, "status": f"ERROR: {e}"})
    print(f"  ✗ Image OCR: {e}")



  --- Source ④: Document Scan (JPG → OCR) ---

2026-03-17 19:24:21.479315: I scikitplot.corpus._readers._image 133692742501952 _image.py:422:get_raw_chunks] ImageReader: opening WHO-EURO-2025-12555-52329-80560-eng.pdf.jpg (backend=tesseract).
2026-03-17 19:24:22.321127: I scikitplot.corpus._readers._image 133692742501952 _image.py:473:get_raw_chunks] ImageReader: finished WHO-EURO-2025-12555-52329-80560-eng.pdf.jpg — 1 frame(s) processed.
2026-03-17 19:24:22.322426: I scikitplot.corpus._base 133692742501952 _base.py:836:get_documents] WHO-EURO-2025-12555-52329-80560-eng.pdf.jpg: yielded 1 documents, omitted 0 (filter/empty).
  ✓ Image OCR: 1 chunks ingested
  [  0] doc_id=f94adbc4a2e8…  source_type=image
        text: 'td pais Gan people afford topay for health care?  New evidence on financial protection  InGreece! su'…
        confidence: 0.592
        page: 0


In [15]:
docs[0].text

'td pais\nGan people afford\ntopay for health care?\n\nNew evidence on\nfinancial protection\n\nInGreece! summary\n\n'

In [16]:

# --- Source ⑤: Audio Podcast (MP3 → ASR) ---
mini_banner("Source ⑤: Audio Podcast (MP3 → ASR)")
audio_path = Path("can-people-afford-to-pay-for-health-care.mp3")
try:
    if audio_path.exists():
        # TODO: Not implemented audio from local
        reader = DocumentReader.create(
            audio_path,
            source_type=SourceType.AUDIO,
            source_title="Can people afford to pay for health care? (podcast)",
            collection_id="who-greece-financial-protection",
            transcribe=True,
            # If Needed like (animal sounds)
            # [ERROR: AudioReader: classify=True requires a 'classifier' callable. Provide a function with signature: classifier(path: Path, offset: float, duration: float) -> list[dict[str, Any]].]
            # classify=True,
            # classifier=my_classifier,
        )
    else:
        # TODO: Not implemented audio from DocumentReader.from_url audio file url link
        # https://archive.org/details/makingcon-241016/makingcon-241016_promo.mp3
	    # https://www.bbc.com/audio/play/w3ct6vk6
        reader = DocumentReader.from_url(
            "https://archive.org/download/makingcon-241016/makingcon-241016_promo.mp3",
            transcribe=True,
        )
        pass
    docs = list(reader.get_documents())
    all_documents.extend(docs)
    source_log.append({"type": "audio_asr", "n_docs": len(docs), "status": "OK"})
    print(f"  ✓ Audio ASR: {len(docs)} chunks ingested")
    if docs:
        show_doc(docs[0], 0)
except ImportError as e:
    source_log.append({"type": "audio_asr", "n_docs": 0, "status": f"SKIP (dep missing)"})
    print(f"  ⚠ Audio ASR skipped (dependency not installed): {type(e).__name__}")
    print(f"    In production: pip install faster-whisper librosa")
except Exception as e:
    source_log.append({"type": "audio_asr", "n_docs": 0, "status": f"ERROR: {e}"})
    print(f"  ✗ Audio ASR: {e}")



  --- Source ⑤: Audio Podcast (MP3 → ASR) ---

2026-03-17 19:24:23.460776: I scikitplot.corpus._readers._audio 133692742501952 _audio.py:1381:get_raw_chunks] AudioReader: no companion found for can-people-afford-to-pay-for-health-care.mp3; transcribing with Whisper model='base'.
2026-03-17 19:24:29.829073: I scikitplot.corpus._readers._audio 133692742501952 _audio.py:640:_transcribe_whisper] AudioReader: transcribing with faster-whisper (model=base).


[2026-03-17 19:24:31.383] [ctranslate2] [thread 28304] [warning] The compute type inferred from the saved model is float16, but the target device or backend do not support efficient float16 computation. The model weights have been automatically converted to use the float32 compute type instead.


2026-03-17 19:26:37.898503: I scikitplot.corpus._readers._audio 133692742501952 _audio.py:1392:get_raw_chunks] AudioReader: transcription produced 55 segments for can-people-afford-to-pay-for-health-care.mp3.
2026-03-17 19:26:37.905684: I scikitplot.corpus._base 133692742501952 _base.py:836:get_documents] can-people-afford-to-pay-for-health-care.mp3: yielded 53 documents, omitted 2 (filter/empty).
  ✓ Audio ASR: 53 chunks ingested
  [  0] doc_id=3dfdf254942f…  source_type=audio
        text: 'Can people afford to pay for healthcare in Europe?'…
        timecode: 0.0s – 6.0s
        confidence: 0.869


In [17]:
docs[0].text

'Can people afford to pay for healthcare in Europe?'

In [ ]:
# TODO: WHO-EURO-2025-12555-52329-80560-eng.zip

In [18]:

# --- Ingestion Summary ---
mini_banner("Ingestion Summary")
for entry in source_log:
    status = "✓" if entry["status"] == "OK" else "⚠"
    print(f"  {status} {entry['type']:25s} → {entry['n_docs']:3d} docs  [{entry['status']}]")
print(f"\n  Total documents in corpus: {len(all_documents)}")



  --- Ingestion Summary ---

  ✓ web_article               →  93 docs  [OK]
  ✓ youtube_transcript        → 101 docs  [OK]
  ✓ pdf_report                →   6 docs  [OK]
  ✓ image_ocr                 →   1 docs  [OK]
  ✓ audio_asr                 →  53 docs  [OK]

  Total documents in corpus: 254


In [19]:

# ===========================================================================
# PHASE 2: NORMALIZE — Clean text for embedding quality
# ===========================================================================

banner("PHASE 2: NORMALIZE — Unicode + Whitespace cleanup")

normalizer = NormalizationPipeline([
    UnicodeNormalizer(),
    WhitespaceNormalizer(),
])
all_documents = normalizer.normalize_batch(all_documents)

n_normalised = sum(1 for d in all_documents if d.normalized_text)
print(f"  ✓ Normalised {n_normalised}/{len(all_documents)} documents")
if all_documents:
    d = all_documents[0]
    print(f"  Example (doc 0):")
    print(f"    text[:80]:           {d.text[:80]!r}")
    print(f"    normalized_text[:80]: {(d.normalized_text or '')[:80]!r}")



  PHASE 2: NORMALIZE — Unicode + Whitespace cleanup

  ✓ Normalised 254/254 documents
  Example (doc 0):
    text[:80]:           'Out-of-pocket payments for primary health care unaffordable for millions in Euro'
    normalized_text[:80]: 'Out-of-pocket payments for primary health care unaffordable for millions in Euro'


In [20]:


# ===========================================================================
# PHASE 3: ENRICH — Tokenize + keywords for KEYWORD/BM25 search
# ===========================================================================

banner("PHASE 3: ENRICH — Tokens + Keywords (NLPEnricher)")

enricher = NLPEnricher(config=EnricherConfig(
    tokenizer="simple",
    keyword_extractor="frequency",
    max_keywords=15,
    remove_stopwords=True,
    min_token_length=3,
))
all_documents = enricher.enrich_documents(all_documents)

n_enriched = sum(1 for d in all_documents if d.tokens)
print(f"  ✓ Enriched {n_enriched}/{len(all_documents)} documents")
if all_documents:
    d = all_documents[0]
    print(f"  Example (doc 0):")
    print(f"    tokens({len(d.tokens or [])}): {(d.tokens or [])[:10]}…")
    print(f"    keywords: {(d.keywords or [])[:8]}")



  PHASE 3: ENRICH — Tokens + Keywords (NLPEnricher)

2026-03-17 19:27:02.999182: I scikitplot.corpus._enrichers._nlp_enricher 133692742501952 _nlp_enricher.py:375:enrich_documents] NLPEnricher: enriched=254, total=254
  ✓ Enriched 254/254 documents
  Example (doc 0):
    tokens(11): ['pocket', 'payments', 'primary', 'health', 'care', 'unaffordable', 'millions', 'europe', 'new', 'report']…
    keywords: ['pocket', 'payments', 'primary', 'health', 'care', 'unaffordable', 'millions', 'europe']


In [21]:

# ===========================================================================
# PHASE 4: INDEX — Build SimilarityIndex (KEYWORD mode, no embeddings)
# ===========================================================================

banner("PHASE 4: INDEX — Build SimilarityIndex (BM25 keyword mode)")

index = SimilarityIndex(config=SearchConfig(match_mode="keyword", top_k=5))
index.build(all_documents)
print(f"  ✓ Index built: {index.n_documents} documents, dense={index.has_embeddings}")



  PHASE 4: INDEX — Build SimilarityIndex (BM25 keyword mode)

2026-03-17 19:27:09.406904: I scikitplot.corpus._similarity._similarity 133692742501952 _similarity.py:331:build] SimilarityIndex: built with 254 documents (dense=False, sparse=True)
  ✓ Index built: 254 documents, dense=False


In [22]:


# ===========================================================================
# PHASE 5: SEARCH — Multi-mode queries
# ===========================================================================

banner("PHASE 5: SEARCH — Multi-mode queries against the corpus")

queries = [
    ("catastrophic health spending Greece", "keyword"),
    ("poorest households", "keyword"),
    ("out-of-pocket payments medicines", "keyword"),
    ("dental care", "strict"),
    ("WHO Barcelona", "strict"),
]

for query, mode in queries:
    mini_banner(f'Search: "{query}" (mode={mode})')
    cfg = SearchConfig(match_mode=mode, top_k=3)
    results = index.search(query, config=cfg)
    if not results:
        print("  (no results)")
    for i, res in enumerate(results):
        text_preview = res.doc.text[:90].replace("\n", " ")
        src = res.doc.source_title or res.doc.source_file
        print(f"  [{i+1}] score={res.score:.4f}  src={src}")
        print(f"      {text_preview!r}…")




  PHASE 5: SEARCH — Multi-mode queries against the corpus


  --- Search: "catastrophic health spending Greece" (mode=keyword) ---

  [1] score=10.5660  src=Can people afford to pay for health care? (podcast)
      'impoverishing health spending and catastrophic health spending.'…
  [2] score=10.3359  src=https://youtu.be/rwPISgZcYIk
      'Health spending and catastrophic Health'…
  [3] score=9.9000  src=https://youtu.be/rwPISgZcYIk
      'we see that catastrophic Health spending'…

  --- Search: "poorest households" (mode=keyword) ---

  [1] score=5.7137  src=https://youtu.be/rwPISgZcYIk
      'look at the poorest fifths of the'…
  [2] score=5.6016  src=https://www.who.int/europe/news/item/12-12-2023-out-of-pocket-payments-for-primary-health-care-unaffordable-for-millions-in-europe-new-who-report-shows
      'Out-of-pocket payments lead to catastrophic health spending for between 1% and 20% of hous'…
  [3] score=4.6102  src=Can people afford to pay for health care? (podcast)
      '

In [23]:

# ===========================================================================
# PHASE 6: ADAPTERS — Export to every downstream format
# ===========================================================================

banner("PHASE 6: ADAPTERS — Export to LangChain / MCP / RAG / LangGraph / HF")

# --- 6a: LangChain Documents ---
mini_banner("6a: LangChain Documents")
lc_docs = to_langchain_documents(all_documents)
first = lc_docs[0]
if isinstance(first, dict):
    print(f"  ✓ {len(lc_docs)} LangChain docs (dict fallback — langchain not installed)")
    print(f"    keys: {list(first.keys())}")
    print(f"    page_content[:80]: {first['page_content'][:80]!r}")
    print(f"    metadata keys: {sorted(first['metadata'].keys())[:10]}")
else:
    print(f"  ✓ {len(lc_docs)} LangChain Document objects")

# --- 6b: LangGraph State ---
mini_banner("6b: LangGraph State")
state = to_langgraph_state(
    all_documents,
    query="catastrophic health spending",
    match_mode="keyword",
)
print(f"  ✓ LangGraph state dict:")
print(f"    keys: {sorted(state.keys())}")
print(f"    n_results: {state['n_results']}")
print(f"    query: {state['query']!r}")

# --- 6c: MCP Resources ---
mini_banner("6c: MCP Resources (Model Context Protocol)")
resources = to_mcp_resources(all_documents[:3])
for r in resources[:2]:
    print(f"  resource:")
    print(f"    uri:      {r['uri']}")
    print(f"    name:     {r['name']}")
    print(f"    mimeType: {r['mimeType']}")
    print(f"    text[:60]: {r['text'][:60]!r}…")

# --- 6d: MCP Tool Result ---
mini_banner("6d: MCP Tool Result (tools/call response)")
tool_result = to_mcp_tool_result(all_documents[:3])
print(f"  ✓ MCP tool result:")
print(f"    isError: {tool_result['isError']}")
print(f"    content items: {len(tool_result['content'])}")
for item in tool_result["content"][:2]:
    print(f"    [{item['type']}] text[:60]: {item['text'][:60]!r}…")
    print(f"         annotations: {item['annotations']}")

# --- 6e: MCP Server (adapter class) ---
mini_banner("6e: MCP Server Adapter")
mcp_server = MCPCorpusServer(index=index, server_name="who-corpus")
tools = mcp_server.list_tools()
print(f"  ✓ MCPCorpusServer: {mcp_server}")
print(f"    tools: {[t['name'] for t in tools]}")
print(f"    tool schema: {json.dumps(tools[0]['inputSchema'], indent=6)[:200]}…")

# --- 6f: HuggingFace Dataset ---
mini_banner("6f: HuggingFace Dataset")
hf = to_huggingface_dataset(all_documents)
if isinstance(hf, dict):
    print(f"  ✓ HuggingFace column dict (datasets lib not installed)")
    print(f"    columns: {sorted(hf.keys())}")
    print(f"    rows: {len(hf['text'])}")
else:
    print(f"  ✓ HuggingFace Dataset: {hf}")

# --- 6g: RAG Tuples ---
mini_banner("6g: RAG Tuples (text, metadata, embedding)")
tuples = to_rag_tuples(all_documents[:3])
for i, (text, meta, emb) in enumerate(tuples):
    print(f"  [{i}] text[:50]: {text[:50]!r}")
    print(f"      meta keys: {sorted(meta.keys())[:8]}")
    print(f"      embedding: {type(emb).__name__}")

# --- 6h: JSONL ---
mini_banner("6h: JSONL Streaming")
lines = list(to_jsonl(all_documents[:3]))
print(f"  ✓ {len(lines)} JSONL lines")
for i, line in enumerate(lines[:2]):
    obj = json.loads(line)
    print(f"  [{i}] keys: {sorted(obj.keys())[:8]}…  text[:50]: {obj['text'][:50]!r}")




  PHASE 6: ADAPTERS — Export to LangChain / MCP / RAG / LangGraph / HF


  --- 6a: LangChain Documents ---

2026-03-17 19:28:41.790406: I scikitplot.corpus._adapters 133692742501952 _adapters.py:182:to_langchain_documents] langchain_core not installed; returning plain dicts with 'page_content' and 'metadata' keys.
  ✓ 254 LangChain docs (dict fallback — langchain not installed)
    keys: ['page_content', 'metadata']
    page_content[:80]: 'Out-of-pocket payments for primary health care unaffordable for millions in Euro'
    metadata keys: ['char_end', 'char_start', 'chunk_index', 'chunking_strategy', 'doc_id', 'element_index', 'html_tag', 'section_type', 'source_file', 'source_type']

  --- 6b: LangGraph State ---

2026-03-17 19:28:41.799244: I scikitplot.corpus._adapters 133692742501952 _adapters.py:182:to_langchain_documents] langchain_core not installed; returning plain dicts with 'page_content' and 'metadata' keys.
  ✓ LangGraph state dict:
    keys: ['documents', 'match_mode', 'n

In [24]:

# ===========================================================================
# PHASE 7: SHOW REAL URL USAGE (code-only, not executed)
# ===========================================================================

banner("PHASE 7: PRODUCTION USAGE — All 5 Real Sources")

PRODUCTION_CODE = '''
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PRODUCTION: Process all 5 real sources via CorpusBuilder
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from scikitplot.corpus import CorpusBuilder, BuilderConfig

builder = CorpusBuilder(BuilderConfig(
    chunker="paragraph",
    normalize=True,
    normalizer_steps=["unicode", "whitespace"],
    enrich=True,
    enricher_kwargs={"keyword_extractor": "frequency", "max_keywords": 15},
    embed=True,                    # requires: pip install sentence-transformers
    embedding_model="all-MiniLM-L6-v2",
    build_index=True,
    collection_id="who-greece-financial-protection",
    source_author="WHO Regional Office for Europe",
))

# ① Web Article (HTML)
result = builder.build([
    "https://www.who.int/europe/news/item/12-12-2023-out-of-pocket-"
    "payments-for-primary-health-care-unaffordable-for-millions-in-"
    "europe-new-who-report-shows",
])

# ② YouTube Video
result = builder.build([
    "https://youtu.be/rwPISgZcYIk?si=TddW-0bwBRF_4_qU",
])

# ③ PDF (download first, then process locally)
import urllib.request
urllib.request.urlretrieve(
    "https://iris.who.int/server/api/core/bitstreams/"
    "7ad66865-7f23-4485-8cf5-7b3d78bdf4f9/content",
    "who_greece_report.pdf",
)
result = builder.build(["who_greece_report.pdf"])

# ④ Image scan (local file or downloaded URL)
result = builder.build(["WHO-EURO-2025-12555-52329-80560-eng_pdf.jpg"])

# ⑤ Audio podcast (local MP3)
result = builder.build(["can-people-afford-to-pay-for-health-care.mp3"])

# ━━━ Or process ALL at once: ━━━
result = builder.build([
    "https://www.who.int/europe/news/item/12-12-2023-out-of-pocket-...",
    "https://youtu.be/rwPISgZcYIk",
    "who_greece_report.pdf",
    "scan.jpg",
    "podcast.mp3",
])

# ━━━ Search ━━━
results = builder.search("catastrophic health spending", match_mode="hybrid")
for r in results:
    print(f"  score={r.score:.4f}  {r.doc.text[:80]}")

# ━━━ Export to ANY consumer ━━━
lc_docs = builder.to_langchain()           # LangChain
state   = builder.to_langgraph_state()     # LangGraph
mcp     = builder.to_mcp_tool_result("...")  # MCP server
hf      = builder.to_huggingface()         # HuggingFace
rag     = builder.to_rag_tuples()          # Vector store
retriever = builder.to_langchain_retriever()  # LangChain retriever
server  = builder.to_mcp_server()          # Full MCP server
builder.export("corpus.parquet")           # File export
'''

print(textwrap.dedent(PRODUCTION_CODE))



  PHASE 7: PRODUCTION USAGE — All 5 Real Sources


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PRODUCTION: Process all 5 real sources via CorpusBuilder
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from scikitplot.corpus import CorpusBuilder, BuilderConfig

builder = CorpusBuilder(BuilderConfig(
    chunker="paragraph",
    normalize=True,
    normalizer_steps=["unicode", "whitespace"],
    enrich=True,
    enricher_kwargs={"keyword_extractor": "frequency", "max_keywords": 15},
    embed=True,                    # requires: pip install sentence-transformers
    embedding_model="all-MiniLM-L6-v2",
    build_index=True,
    collection_id="who-greece-financial-protection",
    source_author="WHO Regional Office for Europe",
))

# ① Web Article (HTML)
result = builder.build([
    "https://www.who.int/europe/news/item/12-12-2023-out-of-pocket-"
    "payments-for-primary-health-care-unaffordable-for-millions-in-"
    "europe-new-who-report-shows",
])

# 

In [25]:

# ===========================================================================
# SUMMARY
# ===========================================================================

banner("SUMMARY")

print(f"  Sources processed:  {len(source_log)}")
print(f"  Total documents:    {len(all_documents)}")
print(f"  Normalised:         {n_normalised}")
print(f"  Enriched (tokens):  {n_enriched}")
print(f"  Index documents:    {index.n_documents}")
print(f"  Dense embeddings:   {index.has_embeddings}")
print()
print("  Adapter outputs demonstrated:")
print("    ✓ LangChain Documents      → to_langchain_documents()")
print("    ✓ LangGraph State          → to_langgraph_state()")
print("    ✓ MCP Resources            → to_mcp_resources()")
print("    ✓ MCP Tool Result          → to_mcp_tool_result()")
print("    ✓ MCP Server Adapter       → MCPCorpusServer()")
print("    ✓ HuggingFace Dataset      → to_huggingface_dataset()")
print("    ✓ RAG Vector Store Tuples  → to_rag_tuples()")
print("    ✓ JSONL Streaming          → to_jsonl()")
print()
print("  Source types supported:")
for st in SourceType:
    print(f"    • {st.value}")
print()
print("  Pipeline complete. All 5 source types → unified corpus → any consumer.")



  SUMMARY

  Sources processed:  5
  Total documents:    254
  Normalised:         254
  Enriched (tokens):  254
  Index documents:    254
  Dense embeddings:   False

  Adapter outputs demonstrated:
    ✓ LangChain Documents      → to_langchain_documents()
    ✓ LangGraph State          → to_langgraph_state()
    ✓ MCP Resources            → to_mcp_resources()
    ✓ MCP Tool Result          → to_mcp_tool_result()
    ✓ MCP Server Adapter       → MCPCorpusServer()
    ✓ HuggingFace Dataset      → to_huggingface_dataset()
    ✓ RAG Vector Store Tuples  → to_rag_tuples()
    ✓ JSONL Streaming          → to_jsonl()

  Source types supported:
    • book
    • article
    • research
    • movie
    • subtitle
    • play
    • poem
    • biography
    • web
    • wiki
    • image
    • video
    • audio
    • spreadsheet
    • code
    • unknown

  Pipeline complete. All 5 source types → unified corpus → any consumer.
